## Image Descriptions with Gemini 

Generate detailed textual descriptions for extracted images using Gemini 2.5 Flash.

**Prerequisites:**
- Make sure you rag-data dir with extracted dir like markdown, images and tables
- Google API key set in .env file

**Output:**
- Markdown descriptions saved to `data/rag-data/images_desc/{company}/{document}/page_X.md`

### Setup and Imports

In [1]:
from dotenv import load_dotenv
load_dotenv()

from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

from PIL import Image

import base64
import io

### Configuration

In [2]:
# Paths
IMAGES_DIR = "industrial_data/images"
OUTPUT_DESC_DIR = "industrial_data/images_desc"

# Model configuration
MODEL_NAME = "gemini-3.1-flash-lite-preview"

model = ChatGoogleGenerativeAI(model=MODEL_NAME)

In [4]:
model.invoke("你好")

AIMessage(content=[{'type': 'text', 'text': '你好！很高兴见到你。请问有什么我可以帮你的吗？', 'extras': {'signature': 'EjQKMgG+Pvb7+u05ZUP4DdvRbzKU7f2cKAmUhcy6glvPCsYnp0kcatyaU/8ZfTtWdvRQNHVn'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d151b-b7de-7d30-adaf-bd941c407b18-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 15, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}})

### Description Generation Function

In [5]:
describe_image_prompt = """请用中文分析这张工业自动化产品合规文档的图片，并以简练的格式提取关键信息。

针对图表与示意图：
- 识别测量维度： 明确图中涉及的物理量（如电压、电流、压力）、安全距离或时序关系。
- 提取关键阈值： 列出曲线图或趋势图中的关键拐点、限值（Limit Lines）及对应的数值。
- 分析物理趋势： 说明指标随环境条件（如温度、海拔、频率）变化的规律（如降额曲线）。

针对表格：
- 核心参数提取： 提取列标题（如试验项目、标准要求、测量值）及关键行数据。
- 合规判定： 重点记录‘判定标准’、‘允许偏差’以及‘试验结果’。
- 注脚信息： 捕获表格下方的特殊限制条件或排除条款。

针对文字说明：
- 安全指令与等级： 提取安全完整性等级 (SIL)、性能等级 (PL) 或防护等级 (IP) 等核心指标。
- 关键事实与数据： 仅汇总具体数值、公差、材料要求和测试循环次数。

要求： 语气客观专业。聚焦于技术指标、安全边界和合规依据，确保提取的信息可直接用于报告汇总或技术比对。"""

In [6]:
from langchain.messages import SystemMessage


def generate_image_description(image_path: Path):
    image = Image.open(image_path)
    buffered = io.BytesIO()
    image.save(buffered, format='PNG')

    image_base64 = base64.b64encode(buffered.getvalue()).decode()

    message = HumanMessage(
        content=[
            {'type': 'text', 'text': describe_image_prompt},
            {'type': 'image_url', 'image_url': f"data:image/png;base64,{image_base64}"}
        ]
    )
    system_prompt = SystemMessage('You are an AI Assistant')

    response = model.invoke([system_prompt, message])

    return response.text

In [7]:
image_path = Path(r'industrial_data\images\Industrial_Automation_Safety_Part02_Pressure_Transmitter\page_17.png')

response = generate_image_description(image_path)

In [9]:
print(response)

以下是基于GB 30439.2—2013标准中“铰接式试验指（GB/T 16842—2008的试具B）”的分析报告：

### 1. 测量维度与结构特征
*   **总长规格**：主体杆身总长为 180 mm。
*   **关键外径**：圆柱杆直径为 $\phi 12$ mm，顶部圆盘直径为 $\phi 50$ mm，厚度为 $5 \pm 0.5$ mm。
*   **铰接设计**：包含三个关节节点（P1、P2、P3），支持在同一平面内弯曲。
*   **几何特征**：
    *   前端球形半径：$R4 \pm 0.05$ mm。
    *   末端锥角：$37^\circ$。
    *   其他圆角：$R2 \pm 0.05$ mm。

### 2. 关键公差与精度要求
标准明确了未标注尺寸的公差执行准则：
*   **角度公差**：$0^\circ / -10'$。
*   **线性尺寸公差**：
    *   $\leqslant 25$ mm：$-0.05$ mm。
    *   $> 25$ mm：$\pm 0.2$ mm。
*   **弯曲限值**：关节设计的弯曲角度应保证在 $90^\circ^{+10^\circ} / 0^\circ$ 范围内。

### 3. 材料与制造工艺
*   **材质要求**：试验指须采用经热处理的钢材制作，以保证在测试过程中的结构强度。
*   **结构约束**：为确保测试准确性，规定关节仅允许在同一平面内弯曲；图中显示的销和槽仅为实现弯曲限制的一种示例手段，实际设计须满足上述角度要求。

### 4. 技术说明与合规要点
*   **应用标准**：本图具对应 GB/T 16842—2008 中的试具B，常用于工业自动化设备的防触电或防机械损伤测试（如手指触及危险部件的安全性验证）。
*   **组件识别**：
    *   1: 绝缘材料；
    *   4: 手柄；
    *   5: 挡板；
    *   6: 球形接触端。
*   **设计原则**：图纸定义的尺寸和公差体系旨在保证该模拟手指在各种工业环境测试中的一致性与可复现性。

---
**核心结论：** 该文档定义了一种标准铰接式测试探针（试具B），重点在于严格控制关节弯曲角度（$90^\circ$）和前端球形尺寸（$R4$），确保其符

In [10]:
# print(response)

def generate_and_save_description(image_path: Path):
    company_name = image_path.parent.parent.name
    doc_name = image_path.parent.name

    output_dir = Path(OUTPUT_DESC_DIR)/company_name/doc_name
    output_dir.mkdir(parents=True, exist_ok=True)

    desc_file = output_dir / f"{image_path.stem}.md"

    if desc_file.exists():
        return False
    
    description = generate_image_description(image_path)
    desc_file.write_text(description, encoding='utf-8')
    
    return True

In [24]:
image_path = Path(r'data\rag-data\images\meta\meta 10-k 2024\page_64.png')

response = generate_and_save_description(image_path)

In [11]:
from tqdm import tqdm

images_path = Path(IMAGES_DIR)
image_files = list(images_path.rglob("page_*.png"))

for image_path in tqdm(image_files):
    response = generate_and_save_description(image_path)


100%|██████████| 31/31 [02:54<00:00,  5.64s/it]
